# 로컬 RGB 로더 최소 CPU 검사 — 한 번 실행

새 Colab **CPU** 런타임에서 위→아래로 실행합니다. 소스 ZIP 한 개만 업로드하며 실제 RGB/pose·manifest·Drive 인증·GPU·128MiB 캐시는 필요 없습니다. 합성 PNG만 임시 경로에 만듭니다. 기존 warm192는 별도 대기입니다.

1. 고정 소스 ZIP 업로드 2. 환경 확인 3. 단일 CPU 검사 4. 보고서 ZIP 다운로드입니다. 같은 런타임의 검사 재실행은 거부합니다. 실패·timeout도 마지막 셀에서 자료를 회수하고 종료합니다. 코드·예산을 바꾸거나 같은 시험을 자동 반복하지 않습니다.

검사 subprocess 상한120초, 각 임시 fixture128KiB·보고서256KiB·stdout/stderr 각각64KiB·결과 ZIP1MiB입니다. 시험 산출물과 fixture의 파일 bytes 합은2MiB 이내로 계획합니다(프로세스 RSS 상한 아님). 소스 업로드64MiB/추출128MiB·환경·Pillow 준비 시간은 별도입니다. Fixture는 각 case 후 제거하고 timeout 때 남더라도 실제 데이터를 건드리지 않습니다. 전체 회귀·학습·속도 벤치마크는 실행하지 않습니다.

**준비 시점에는 스크립트/노트북의 정적 검토만 했으며 통과 결과는 없습니다.** 소스 revision·ZIP hash와 보고서의 고정 로더 hash는 테스트 세션의 인계값과 대조합니다. Pillow가 없으면 환경 셀에 표시한 Pillow 단독 설치 안내를 따른 뒤 검사 시작 전에만 환경 셀을 다시 확인합니다. Torch가 없으면 중단하며 자동 설치하지 않습니다.


In [ ]:
import sys, os, io, json, stat, re, hashlib, tempfile, zipfile, shutil
from pathlib import Path, PurePosixPath
from google.colab import files

if "RGB_RUN_ROOT" in globals():
    raise RuntimeError("이미 준비한 실행입니다. 기존 마지막 셀에서 ZIP을 보관하세요. 자동 재실행하지 않습니다.")
RGB_RUN_ROOT = Path(tempfile.mkdtemp(prefix="rewind_rgb_check_", dir="/content"))
SOURCE_OK = False
PROJECT_ROOT = None
source_receipt = {"status":"failed", "run_root":str(RGB_RUN_ROOT)}
try:
    uploaded = files.upload()  # 테스트 세션에서 받은 코드 source ZIP 한 개
    if len(uploaded) != 1:
        raise ValueError("코드 source ZIP 한 개가 필요합니다.")
    source_name, source_bytes = next(iter(uploaded.items()))
    if not source_name.lower().endswith(".zip") or len(source_bytes)>64*1024**2:
        raise ValueError("64MiB 이하의 코드 source ZIP이 필요합니다.")
    source_receipt.update(name=source_name, bytes=len(source_bytes), sha256=hashlib.sha256(source_bytes).hexdigest())
    excluded={".git",".venv","data","outputs","checkpoints",".local-archive",".aws",".codex",".agents","__pycache__"}
    with zipfile.ZipFile(io.BytesIO(source_bytes)) as archive:
        members=archive.infolist()
        if len(members)>5000 or sum(m.file_size for m in members)>128*1024**2:
            raise ValueError("코드 ZIP의 항목 수/해제 크기 상한을 초과했습니다.")
        roots=[PurePosixPath(m.filename).parent.parent for m in members if m.filename.endswith("rewind_m1/local_rgb.py")]
        if len(roots)!=1:
            raise ValueError("local_rgb.py와 tests/check_local_rgb.py가 있는 고정 source ZIP이 필요합니다. 결과/배포 묶음 ZIP을 대신 넣지 마세요.")
        seen=set()
        for m in members:
            p=PurePosixPath(m.filename)
            mode=stat.S_IFMT(m.external_attr>>16)
            if (p.is_absolute() or ".." in p.parts or "\\" in m.filename or not p.parts
                or any(":" in x or x.endswith(("."," ")) for x in p.parts)
                or any(ord(c)<32 for c in m.orig_filename) or mode not in (0,stat.S_IFREG,stat.S_IFDIR)
                or m.flag_bits&1):
                raise ValueError("안전하지 않은 코드 ZIP 경로/유형입니다.")
            key=p.as_posix().casefold().rstrip("/")
            if key in seen:raise ValueError("중복 코드 ZIP 경로입니다.")
            seen.add(key)
        extract_root=RGB_RUN_ROOT/"source"
        extract_root.mkdir()
        for m in members:
            parts=PurePosixPath(m.filename).parts
            if not any(p in excluded or p==".env" or p.startswith(".env.") for p in parts):
                archive.extract(m,extract_root)
    PROJECT_ROOT=(extract_root/roots[0]).resolve()
    if not PROJECT_ROOT.is_relative_to(extract_root.resolve()):raise ValueError("source root 이탈")
    for relative in ("tests/check_local_rgb.py","rewind_m1/local_rgb.py","SOURCE_REVISION.txt"):
        if not (PROJECT_ROOT/relative).is_file():raise ValueError("고정 소스 파일 누락: "+relative)
    source_revision=(PROJECT_ROOT/"SOURCE_REVISION.txt").read_text(encoding="utf-8").strip()
    if not re.fullmatch(r"[0-9a-fA-F]{40}",source_revision):raise ValueError("고정 source revision이 필요합니다.")
    source_receipt.update(status="ready", source_revision=source_revision, project_root=str(PROJECT_ROOT))
    SOURCE_OK=True
except Exception as exc:
    source_receipt.update(exception_type=type(exc).__name__, reason=str(exc)[:1200])
finally:
    (RGB_RUN_ROOT/"source_upload.json").write_text(json.dumps(source_receipt,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
    for variable in ("uploaded","source_bytes"):
        globals().pop(variable,None)
print(source_receipt)


In [ ]:
import importlib.util, importlib.metadata, platform
if globals().get("RGB_TEST_STARTED",False):
    raise RuntimeError("검사가 이미 시작됐습니다. 결과를 보관하고 종료하세요.")
environment={"python":platform.python_version(),"platform":platform.platform(),"source_ready":SOURCE_OK}
for package,module in (("torch","torch"),("Pillow","PIL")):
    present=importlib.util.find_spec(module) is not None
    try:version=importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:version=None
    environment[package]={"present":present,"version":version}
ENV_READY=SOURCE_OK and all(environment[p]["present"] for p in ("torch","Pillow"))
environment["ready"]=ENV_READY
(RGB_RUN_ROOT/"environment.json").write_text(json.dumps(environment,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print(environment)
if not environment["Pillow"]["present"]:
    print('Pillow만 없으면 검사 시작 전에 별도 셀에서 %pip install "Pillow>=10.4" 실행 후 이 환경 셀을 확인하세요. 자동 설치하지 않습니다.')
if not environment["torch"]["present"]:
    print("Torch가 없습니다. 이 실행은 중단하고 마지막 셀에서 보고서를 보관하세요. Torch 자동 설치/GPU 변경은 하지 않습니다.")


In [ ]:
import subprocess, time, traceback
if globals().get("RGB_TEST_STARTED",False):
    raise RuntimeError("동일 실행을 반복하지 않습니다. 마지막 셀에서 ZIP을 회수하세요.")
RGB_TEST_STARTED=True
CHECK_OUTPUT=RGB_RUN_ROOT/"check"
started=time.monotonic()
supervision={"status":"failed","timeout_seconds":120,"test_started":False}
stdout=stderr=""
try:
    if not ENV_READY:raise RuntimeError("환경/source 준비 실패: 검사를 실행하지 않았습니다.")
    command=[sys.executable,"-B",str(PROJECT_ROOT/"tests/check_local_rgb.py"),"--output",str(CHECK_OUTPUT)]
    supervision.update(command=command,test_started=True)
    child=subprocess.run(command,cwd=PROJECT_ROOT,capture_output=True,text=True,encoding="utf-8",errors="replace",timeout=120,
                         env={**os.environ,"PYTHONDONTWRITEBYTECODE":"1","CUDA_VISIBLE_DEVICES":"","OMP_NUM_THREADS":"1"})
    stdout,stderr=child.stdout,child.stderr
    supervision["returncode"]=child.returncode
    report=json.loads((CHECK_OUTPUT/"report.json").read_text(encoding="utf-8"))
    supervision["status"]="passed" if child.returncode==0 and report.get("status")=="passed" and report.get("completed_cases")==report.get("required_cases")==12 else "failed"
except subprocess.TimeoutExpired as exc:
    supervision.update(status="timeout",exception_type=type(exc).__name__)
    stdout=exc.stdout or "";stderr=exc.stderr or ""
except Exception as exc:
    supervision.update(status="failed",exception_type=type(exc).__name__,reason=str(exc)[:1200],traceback=traceback.format_exc()[-5000:])
for name,value in (("stdout",stdout),("stderr",stderr)):
    raw=value if isinstance(value,bytes) else value.encode("utf-8",errors="replace")
    (RGB_RUN_ROOT/(name+".log")).write_bytes(raw[:65536])
    if len(raw)>65536:supervision.update(status="failed",log_truncated=True)
if not (CHECK_OUTPUT/"report.json").is_file():
    CHECK_OUTPUT.mkdir(exist_ok=True)
    (CHECK_OUTPUT/"report.json").write_text(json.dumps({"schema_version":"local_rgb_cpu_check_v1","status":"not_completed",
        "reason":supervision["status"],"cases":[],"limitations":"worker report missing; this is not a pass"},indent=2)+"\n",encoding="utf-8")
supervision["elapsed_seconds"]=time.monotonic()-started
(RGB_RUN_ROOT/"supervision.json").write_text(json.dumps(supervision,ensure_ascii=False,indent=2)+"\n",encoding="utf-8")
print(supervision)
print((RGB_RUN_ROOT/"stdout.log").read_text(encoding="utf-8",errors="replace"))
print("성공/실패/timeout 모두 다음 셀에서 ZIP을 보관하고 종료합니다. 자동 재실행하지 않습니다.")


In [ ]:
# Worker 실패/timeout/환경 실패도 네트워크 원본 요청 없이 회수합니다.
result_zip=RGB_RUN_ROOT/"local_rgb_check_result.zip"
if not result_zip.exists():
    entries=[(RGB_RUN_ROOT/p,p) for p in ("source_upload.json","environment.json","supervision.json","stdout.log","stderr.log","check/report.json") if (RGB_RUN_ROOT/p).is_file()]
    if sum(p.stat().st_size for p,_ in entries)>512*1024:
        raise ValueError("보고서/로그 512KiB 상한 초과: 기존 자료를 보존하고 중단하세요.")
    with zipfile.ZipFile(result_zip,"x",compression=zipfile.ZIP_DEFLATED) as bundle:
        for path,name in entries:bundle.write(path,name)
    if result_zip.stat().st_size>1024**2:raise ValueError("결과 ZIP 1MiB 상한 초과")
print({"result_zip":str(result_zip),"bytes":result_zip.stat().st_size,"sha256":hashlib.sha256(result_zip.read_bytes()).hexdigest()})
files.download(str(result_zip))
